# 300 m walkability computation

**Goal:** Identify which residential buildings (street number to be precise for each inhabitant in the city) in Bologna are within a **300m walking distance** of a publicly accessible park or green space.

**Pipeline overview:**
1. Load city boundary, buildings street number, parks, buldings polygons and pedestrian network
2. Detect park entrance points (where the pedestrian network intersects park boundaries)
3. Generate 300m walking isochrones from each entrance using GraphHopper
4. Spatial-join buildings against the dissolved isochrones
5. Compute precise building-to-entrance walk distances for matched buildings
6. Export results as GeoJSON, CSV, and SQLite

---

> **GraphHopper must be running before executing the distance cells.**  
> In a separate terminal, navigate to the folder containing `graphhopper-web-11.0.jar`, `config_bologna.yml`, and `Bologna.osm.pbf`, then run:
> ```bash
> rm -rf graph-cache
> java -jar graphhopper-web-11.0.jar server config_bologna.yml
> ```
> Wait for `Started server at HTTP 8989` before continuing.

In [ ]:
!pip install tqdm

  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
Using cached tqdm-4.67.3-py3-none-any.whl (78 kB)
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [104]:
import warnings
warnings.filterwarnings("ignore")

import os
import pickle
import sqlite3
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import osmnx as ox
import pandas as pd
import requests
from shapely.geometry import Point, shape
from shapely.strtree import STRtree
from tqdm import tqdm
from shapely.ops import nearest_points

os.chdir("/Users/terezasaskova/Library/Mobile Documents/com~apple~CloudDocs/3-30-300-index")

#Coordinate reference systems 
CRS_WGS    = "EPSG:4326"   # WGS-84 geographic (lat/lon) — required by GraphHopper
CRS_METRIC = "EPSG:32632"  # UTM zone 32N — metric distances in metres

#Analysis thresholds 
WALKING_DISTANCE = 300   # metres — maximum acceptable walk to a park entrance
EUCLID_CUTOFF    = 420   # metres — Euclidean pre-filter; keeps API calls minimal
                         # (straight-line distance is always ≤ walking distance,
                         #  so a ~1.4× buffer safely avoids false negatives)

#GraphHopper endpoints 
ISO_URL = "http://localhost:8989/isochrone"
GH_URL  = "http://localhost:8989/route"

## 1 - Load Input Data

Four data sources are needed:
| Layer | Source | info |
|---|---|---|
| Municipal boundary | `bologna_boundary.gpkg` | area of interest |
| Buildings with civic numbers | `bologna_buildings_streetnum.gpkg` | street numbers |
| Green spaces (three sub-layers) | `bologna_greenery.gpkg` | filtered greenery from DSBN and OSM |
| Buildings polygons | `bologna_buildings.gpkg` | buildings polygons |

+  +open street map - pedestrian paths - walking distance 

In [112]:
#1a. Municipal boundary 
bologna_boundary = gpd.read_file("data/bologna_boundary.gpkg").to_crs(CRS_METRIC)

# WGS-84 version needed by osmnx to download the street network
bologna_wgs84 = bologna_boundary.to_crs(CRS_WGS).geometry.union_all()

In [147]:
# 1b. Civici (address points — one point per street number)
civici = (
    gpd.read_file("data/300m/bologna_buildings_streetnum.gpkg")
    .to_crs(CRS_METRIC)
    .loc[lambda df: df.geometry.notna()]
    .reset_index(drop=True)
)
civici["building_id"] = civici["indirizzo_completo"]
print(f"{len(civici):,} civici loaded")

77,630 civici loaded


In [116]:
#1c. Green spaces
# two layers are merged:
#   pe_unis_all — green spaces - settlment units
#   osm_public — parks from OpenStreetMap - w. private tag - filter out

parks = (
    pd.concat(
        [gpd.read_file("data/300m/bologna_greenery.gpkg", layer=l)
         for l in ["pe_unis_all", "osm_public"]],
        ignore_index=True,
    )
    .to_crs(CRS_METRIC)
    .reset_index(drop=True)
    .assign(park_id=lambda df: df.index)  # unique integer key used throughout
)

print(f"{len(parks):,} parks (before filtering)")
print(parks["layer_type"].value_counts())

1,568 parks (before filtering)
layer_type
osm_public     1026
pe_unis_all     542
Name: count, dtype: int64


## 1. Pedestrian network 
- Download the full walk network for Bologna from OSM and project to UTM 32N.
- The edges GeoDataFrame is used to detect park entrances in the next section.

In [117]:
G = ox.graph_from_polygon(bologna_wgs84, network_type="walk")
G = ox.project_graph(G, to_crs=CRS_METRIC)

_, edges = ox.graph_to_gdfs(G)
edges = edges.reset_index(drop=True)

print(f"{len(edges):,} pedestrian edges")

66,806 pedestrian edges


## 2 Park Entrance 

- available entrances (osm) = whenever such information is available in osm 
- no explicit entrance = intersection between points of pedestrian crossings x the park boundary

Steps:
1. Build an STRtree over all edge geometries 
2. For each park, compute `boundary x edges` 

In [118]:
#get entrances from intersection or existing entrance in osm
def extract_points(geom) -> list:
    """
    Recursively extract individual Point objects from any geometry type.
    A boundary-edge intersection can produce a Point, MultiPoint, or
    GeometryCollection (e.g. a LineString that overlaps a wall segment
    plus a Point where a path crosses the boundary).
    """
    if geom is None or geom.is_empty:
        return []
    if geom.geom_type == "Point":
        return [geom]
    if geom.geom_type == "MultiPoint":
        return list(geom.geoms)
    if geom.geom_type == "GeometryCollection":
        pts = []
        for g in geom.geoms:
            pts.extend(extract_points(g))
        return pts
    return []   # LineString / Polygon — not useful as entrance points

In [119]:
# fetch real OSM entrance nodes 
osm_entrances_raw = ox.features_from_polygon(
    bologna_wgs84, tags={"entrance": True}
).to_crs(CRS_METRIC)

osm_entrance_pts = osm_entrances_raw[
    osm_entrances_raw.geometry.geom_type == "Point"
].copy()
print(f"OSM entrance nodes found: {len(osm_entrance_pts)}")

OSM entrance nodes found: 375


## Entrance detection : three fallback steps per park:

1. Explicit OSM entrance=* node within 2 m of park boundary
2. Intersection of park boundary with pedestrian edges
3. Snap park centroid to nearest edge (OSM parks only)

In [ ]:

osm_entrance_tree = STRtree(osm_entrance_pts.geometry.values)
edges_geom = edges.geometry.values
tree = STRtree(edges_geom)


rows = []
step_counts = {1: 0, 2: 0, 3: 0}

for _, park in tqdm(parks.iterrows(), total=len(parks), desc="Park entrances"):
    boundary = park.geometry.boundary
    park_id  = park["park_id"]
    pts      = []

    # Step 1 — explicit OSM entrance
    idxs = osm_entrance_tree.query(park.geometry.buffer(2), predicate="intersects")
    for idx in idxs:
        pt = osm_entrance_pts.geometry.iloc[idx]
        if boundary.distance(pt) <= 2.0:
            pts.append(pt)

    if pts:
        step_counts[1] += 1

    # Step 2 — boundary ∩ pedestrian edges 
    if not pts:
        for idx in tree.query(boundary, predicate="intersects"):
            inter = boundary.intersection(edges_geom[idx])
            pts.extend(extract_points(inter))
        if pts:
            step_counts[2] += 1

    # Step 3 = OSM-sourced parks only: snap centroid to nearest pedestrian edge
    # Only applied to OSM parks 
    if not pts and park["source"] == "OSM":
        centroid = park.geometry.centroid
        nearest_idxs = tree.query(centroid.buffer(150), predicate="intersects")
        if len(nearest_idxs) > 0:
            closest_edge = min(nearest_idxs,
                               key=lambda i: edges_geom[i].distance(centroid))
            snap_pt, _ = nearest_points(edges_geom[closest_edge], centroid)
            pts.append(snap_pt)
            step_counts[3] += 1

    for pt in pts:
        rows.append({"park_id": park_id, "geometry": pt})


entrances = gpd.GeoDataFrame(rows, crs=CRS_METRIC)
missing = set(parks["park_id"]) - set(entrances["park_id"])

print(f"\n{len(entrances):,} raw entrance points")
print(f"Parks resolved by step: OSM nodes={step_counts[1]}, boundary={step_counts[2]}, osm without entrance={step_counts[3]}")
print(f"Parks with no entrance found: {len(missing)}")

Park entrances: 100%|██████████| 1568/1568 [00:13<00:00, 115.81it/s]



6,171 raw entrance points
Parks resolved by step: OSM nodes=14, boundary=531, osm without entrance=703
Parks with no entrance found: 320


In [134]:
num_parks = len(parks)
final = num_parks - missing
print(f"parks: {(num_parks)}")
print(f"filter out: {(missing)}")

print(f"after filtering: {(final)}")


parks: 1568
filter out: 320
after filtering: 1248


## 3 - Walking Isochrones (GraphHopper)

For each entrance point, request a **300 m walking isochrone** from the local GraphHopper server. The isochrone is a polygon covering all locations reachable within 300 m of walking from that entrance.



In [145]:
#GraphHopper isochrone
def get_isochrone(lat, lon, distance_m=300):
    """
    Request a distance-based walking isochrone from GraphHopper.
    Returns the GeoJSON polygon feature, or None on failure.
    """
    params = {
        "point":          f"{lat},{lon}",
        "profile":        "foot",
        "distance_limit": distance_m,
        "type":           "json",
    }
    try:
        r = requests.get(ISO_URL, params=params, timeout=10)
        r.raise_for_status()
        return r.json()["polygons"][0]
    except Exception:
        return None


# Convert entrances to WGS84 for GraphHopper
entrances_wgs = entrances.to_crs(CRS_WGS)
entrances["entrance_lat"] = entrances_wgs.geometry.y
entrances["entrance_lon"] = entrances_wgs.geometry.x

iso_rows = []
for _, row in tqdm(entrances.iterrows(), total=len(entrances), desc="Isochrones"):
    result = get_isochrone(row["entrance_lat"], row["entrance_lon"])
    if result:
        iso_rows.append({
            "park_id":      row["park_id"],
            "entrance_lat": row["entrance_lat"],
            "entrance_lon": row["entrance_lon"],
            "geometry":     shape(result["geometry"]),
        })
 
# Isochrones stay in WGS84 from GH, project to metric for spatial join
isochrones = gpd.GeoDataFrame(iso_rows, crs=CRS_WGS).to_crs(CRS_METRIC)
print(f"{len(isochrones):,} isochrones computed "
      f"({len(entrances) - len(isochrones):,} failed)")


Isochrones: 100%|██████████| 6171/6171 [10:19<00:00,  9.97it/s]  


6,171 isochrones computed (0 failed)


## 4 - Spatial Join: Buildings Within 300 m Walk

Each civico that falls inside multiple overlapping isochrones will produce
- multiple rows — one per (civico, entrance) pair.
- this is intentional: Step 5 routes all pairs and keeps the shortest.

In [148]:
civici_gdf = gpd.GeoDataFrame(
    {"building_id": civici["building_id"]},
    geometry=civici.geometry,   # already Point geometries in CRS_METRIC
    crs=CRS_METRIC,
)
 
joined = gpd.sjoin(
    civici_gdf,
    isochrones[["park_id", "entrance_lat", "entrance_lon", "geometry"]],
    how="left",
    predicate="within",
)
 
# All valid (civico, entrance) candidate pairs
candidates = (
    joined[joined["park_id"].notna()]
    .copy()
    .reset_index(drop=True)
)
 
# WGS84 coordinates of each civico for GraphHopper routing
cands_wgs = gpd.GeoDataFrame(
    candidates[["building_id"]],
    geometry=candidates.geometry,
    crs=CRS_METRIC,
).to_crs(CRS_WGS)
candidates["orig_lat"] = cands_wgs.geometry.y
candidates["orig_lon"] = cands_wgs.geometry.x
 
n_pairs = len(candidates)
n_cands = candidates["building_id"].nunique()
print(f"\n{n_cands:,} candidate civici "
      f"({n_cands / len(civici) * 100:.1f}% of total)")
print(f"{n_pairs:,} (civico, entrance) pairs to route")
print(f"avg {n_pairs / max(n_cands, 1):.1f} entrance candidates per civico")
 


58,568 candidate civici (75.4% of total)
619,358 (civico, entrance) pairs to route
avg 10.6 entrance candidates per civico


## 5 - Route ALL (civico, entrance) pairs — keep shortest walk per civico
- GraphHopper gives the true walking distance along the street network.
- Routing to every candidate entrance (not just one) ensures we find the
- genuinely nearest entrance by walking, not by straight-line distance.

In [149]:
#GraphHopper point-to-point routing call 
def graphhopper(orig_lat, orig_lon, dest_lat, dest_lon):
    """
    Return the shortest walking distance (metres) between two lat/lon points.
    Returns None if the request fails or no path is found.
    """
    params = {
        "point":       [f"{orig_lat},{orig_lon}", f"{dest_lat},{dest_lon}"],
        "profile":     "foot",
        "calc_points": "false",   # geometry not needed — distance only
        "instructions": "false",
    }
    try:
        r = requests.get(GH_URL, params=params, timeout=5)
        r.raise_for_status()
        return r.json()["paths"][0]["distance"]
    except Exception:
        return None

In [150]:
results = []
for _, row in tqdm(candidates.iterrows(), total=n_pairs,
                   desc="Routing civico→entrance"):
    dist = graphhopper(
        row["orig_lat"],     row["orig_lon"],
        row["entrance_lat"], row["entrance_lon"],
    )
    if dist is not None:
        results.append({
            "building_id":  row["building_id"],
            "park_id":      row["park_id"],
            "entrance_lat": row["entrance_lat"],
            "entrance_lon": row["entrance_lon"],
            "walk_dist_m":  dist,
        })
 
results_df = pd.DataFrame(results)
print(f"\n{len(results_df):,} pairs routed successfully "
      f"({n_pairs - len(results_df):,} GH failures)")
 
# Per civico: keep the entrance with the shortest WALKING distance
best = (
    results_df
    .sort_values("walk_dist_m")
    .groupby("building_id", sort=False)
    .first()
    .reset_index()
)

Routing civico→entrance: 100%|██████████| 619358/619358 [1:53:26<00:00, 90.99it/s]  



619,358 pairs routed successfully (0 GH failures)


## 6 -  final output 

In [151]:

# Start from full civici table so every address point appears in the output
out = civici.merge(
    best[["building_id", "park_id", "entrance_lat", "entrance_lon", "walk_dist_m"]],
    on="building_id",
    how="left",
)
 
out["within_300m"]  = out["walk_dist_m"].le(WALKING_DISTANCE)
out["is_candidate"] = out["building_id"].isin(candidates["building_id"])
 
# Attach park attributes
name_col = next(
    (c for c in ["nome", "name", "descrizione"] if c in parks.columns), None
)
park_cols = ["park_id", "area_ha", "source", "layer_type"]
if name_col:
    park_cols.append(name_col)
 
out = out.merge(parks[park_cols].copy(), on="park_id", how="left")
 
# Rename to clean output column names
rename = {
    "ncivsub":            "street_number",
    "indirizzo_completo": "address",
    "quartiere":          "neighbourhood",
    "layer_type":         "park_type",
    "area_ha":            "park_area_ha",
    "source":             "park_source",
    "park_id":            "nearest_park_id",
}
if name_col:
    rename[name_col] = "park_name"
out = out.rename(columns=rename)
 
out = gpd.GeoDataFrame(out, geometry="geometry", crs=CRS_METRIC)
 
# Summary
n_total   = len(out)
n_within  = out["within_300m"].sum()
n_no_gh   = out["is_candidate"] & out["walk_dist_m"].isna()
n_no_iso  = (~out["is_candidate"]).sum()
 
print(f"Total civici:                {n_total:,}")
print(f"Within 300 m walk:           {n_within:,}  ({n_within/n_total*100:.1f}%)")
print(f"Candidate, no GH route:      {n_no_gh.sum():,}")
print(f"No isochrone match at all:   {n_no_iso:,}")


Total civici:                77,630
Within 300 m walk:           56,308  (72.5%)
Candidate, no GH route:      0
No isochrone match at all:   19,062


## 7 - Export Results
output formats are written to `data/`:

File 

- `buildings_300m_indicator.geojson` 
- `buildings_300m_indicator.csv` 
- `300_walkability_map.gpkg` 

In [ ]:
out.to_file("data/300m/300_walkability_map.gpkg",
            driver="GPKG", layer="buildings")
parks[parks["park_id"].isin(out["nearest_park_id"])].to_file(
    "data/300m/300_walkability_map.gpkg", driver="GPKG", layer="parks"
)
 
# GeoJSON + CSV (WGS84)
out_wgs = out.to_crs(CRS_WGS)
out_wgs["longitude"] = out_wgs.geometry.x
out_wgs["latitude"]  = out_wgs.geometry.y
 
final_cols = [
    "street_number", "address", "longitude", "latitude",
    "neighbourhood", "nearest_park_id", "park_type", "park_source",
    "park_area_ha", "entrance_lat", "entrance_lon",
    "walk_dist_m", "within_300m", "geometry",
]
if "park_name" in out_wgs.columns:
    final_cols.insert(final_cols.index("park_type"), "park_name")
 
out_wgs = out_wgs[[c for c in final_cols if c in out_wgs.columns]]
 
out_wgs.to_file("data/buildings_300m_indicator.geojson", driver="GeoJSON")
out_wgs.drop(columns="geometry").to_csv(
    "data/300m/buildings_300m_indicator.csv", index=False
)
 
print("data/300m/300_walkability_map.gpkg  (buildings + parks layers)")
print("data/300m/buildings_300m_indicator.geojson")
print("data/300m/buildings_300m_indicator.csv")
print(out_wgs.head(3).to_string())
 

data/300m/300_walkability_map.gpkg  (buildings + parks layers)
data/buildings_300m_indicator.geojson
data/buildings_300m_indicator.csv
  street_number                             address  longitude   latitude          neighbourhood  nearest_park_id park_name    park_type park_source  park_area_ha  entrance_lat  entrance_lon  walk_dist_m  within_300m                   geometry
0          27/1        VIA MARCO EMILIO LEPIDO 27/1  11.278541  44.516819  Borgo Panigale - Reno            917.0      None   osm_public         OSM      0.081845     44.517077     11.278960       79.759         True  POINT (11.27854 44.51682)
1          46/2  VIA GIOVANNI BATTISTA GANDINO 46/2  11.355349  44.478504          Santo Stefano            216.0       NaN  pe_unis_all        DBSN      4.855096     44.479283     11.354519      111.623         True   POINT (11.35535 44.4785)
2           9/2              VIA GUIDO ZUCCHINI 9/2  11.351607  44.501726          Santo Stefano           1397.0      None   osm_pub